In [ ]:
from __future__ import annotations


class GrowthService:
    """
    Produces project-level growth metrics from persistent
    concept mastery.
    """

    def __init__(self, database):
        self.database = database

    def get_growth(
        self,
        user_id: str,
        project_id: str,
    ) -> dict:

        mastery = list(
            self.database.collection(
                "mastery"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            )
        )

        if not mastery:
            return {
                "project_id": project_id,
                "average_mastery": 0.0,
                "concept_count": 0,
                "improving": 0,
                "stable": 0,
                "needs_attention": 0,
                "strong_concepts": [],
                "weak_concepts": [],
            }

        scores = [
            float(
                item.get(
                    "score",
                    0.0,
                )
            )
            for item in mastery
        ]

        strong_concepts = [
            item.get("concept_id")
            for item in mastery
            if float(
                item.get("score", 0.0)
            ) >= 0.80
        ]

        weak_concepts = [
            item.get("concept_id")
            for item in mastery
            if float(
                item.get("score", 0.0)
            ) < 0.50
        ]

        return {
            "project_id": project_id,
            "average_mastery": (
                sum(scores) / len(scores)
            ),
            "concept_count": len(mastery),
            "improving": sum(
                1
                for item in mastery
                if item.get("trend") == "improving"
            ),
            "stable": sum(
                1
                for item in mastery
                if item.get("trend") == "stable"
            ),
            "needs_attention": sum(
                1
                for item in mastery
                if item.get("trend")
                == "needs_attention"
            ),
            "strong_concepts": strong_concepts,
            "weak_concepts": weak_concepts,
        }